# Drawdowns

VaR and expected shortfall (05/01, 05/02) describe the loss over **one** horizon. An investor
lives through a **path**: a fall of 35% from the last peak, two years under water, then a slow
recovery. The drawdown measures that path. It is the risk number clients remember, the one
hedge-fund mandates stop out on, and the only common risk measure that depends on the **order**
of returns.

## 0. Math link

There is no single course notebook behind drawdowns; the notebook uses tools from several:

- [Cumulative distribution function](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/01_probability_distribution/08_cumulative_distribution_function.ipynb):
  "probability of a drawdown deeper than $x$" is a survival function $1 - F(x)$.
- [Law of large numbers](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/05_law_of_large_numbers.ipynb):
  why a positive drift eventually wins over volatility, and drawdowns stop growing.
- [Derivative of logarithmic functions](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/05_derivative_of_logarithmic_functions.ipynb):
  log wealth turns products of returns into sums.
- The two Brownian-motion results in the Theory section (probability of ever falling $x$ below
  the start, expected maximum drawdown) are stated and checked by simulation, not proved. Their
  derivation needs stochastic processes, a gap in the course repos (module 10).

## 1. Introduction

A fund that returns 7% a year with 16% volatility looks the same in a mean–variance table whether
its bad years are spread out or come back to back. For the investor they are not the same: three
bad years in a row produce a far deeper loss from the peak, and an investor who redeems at the
bottom locks it in.

After this notebook you can:

- compute the drawdown series, the maximum drawdown and the time under water;
- explain why a loss of $d$ needs a gain of $d/(1-d)$ to recover;
- show that drawdowns depend on the order of returns while volatility does not;
- use the Brownian-motion formulas to estimate how likely a drawdown of a given depth is;
- explain why maximum drawdowns of funds with different track-record lengths are not comparable.

## 2. Theory

### Definitions

Start with wealth $W_0 = 1$ and simple returns $r_t$, so $W_t = \prod_{s=1}^t (1 + r_s)$. The
**running peak** is $M_t = \max_{s \le t} W_s$ and the **drawdown** at $t$ is

$$
\mathrm{DD}_t = 1 - \frac{W_t}{M_t} \in [0, 1)
$$

the percentage loss from the highest wealth so far. Over a period $[0, T]$:

- **maximum drawdown** $\mathrm{MDD}_T = \max_{t \le T} \mathrm{DD}_t$, the worst peak-to-trough loss;
- **time under water**: the length of a stretch with $\mathrm{DD}_t > 0$, from a peak to the next new
  peak (or to $T$ if not yet recovered);
- **time to recovery**: from the trough back to the old peak.

### Recovery asymmetry

A fall of $d$ leaves $1 - d$. To get back to 1 requires a gain $g$ with $(1-d)(1+g) = 1$:

$$
g = \frac{d}{1 - d}
$$

| loss $d$ | 10% | 20% | 33% | 50% | 75% | 90% |
|---|---|---|---|---|---|---|
| gain needed $g$ | 11% | 25% | 50% | 100% | 300% | 900% |

### Log drawdown

With log wealth $X_t = \log W_t = \sum_s \log(1 + r_s)$, the log drawdown is the distance below the
running maximum of a sum:

$$
D_t = \max_{s \le t} X_s - X_t = -\log(1 - \mathrm{DD}_t)
$$

So a 50% drawdown is a log drawdown of $\log 2 = 0.693$. If daily log returns are i.i.d. with mean
$m$ and variance $\sigma^2$ (both annualized below), $X_t$ is a random walk with drift, close to a
Brownian motion with drift $m$ and volatility $\sigma$. For returns with arithmetic mean $\mu$, $m \approx \mu - \sigma^2/2$.

### Path dependence

$\mathrm{MDD}$ depends on the order of returns. Mean, volatility, Sharpe ratio, VaR and ES of the return
distribution do not: shuffle the returns and they are unchanged, but the drawdown changes
(Example 3).

### Two Brownian-motion results (stated, checked numerically below)

**1. Probability of ever falling $x$ below the start.** For a Brownian motion with drift $m > 0$ and
volatility $\sigma$:

$$
P\Big(\min_{t \ge 0} (X_t - X_0) \le -x\Big) = e^{-2 m x/\sigma^2}
$$

The same exponential is the long-run distribution of the log drawdown $D_t$: in the long run,
$P(D_t > x) = e^{-2mx/\sigma^2}$, an exponential distribution with mean $\sigma^2/(2m)$. The ratio
$\sigma^2/(2m)$ is the natural drawdown scale: halve the drift or multiply the volatility by $\sqrt 2$, and
typical drawdowns double.

**2. Expected maximum drawdown over $T$ years.** With zero drift (Magdon-Ismail et al., 2004):

$$
E[\mathrm{MDD}_T^{\log}] = \sqrt{\pi/2}\;\sigma\sqrt T \approx 1.25\,\sigma\sqrt T
$$

growing without bound like $\sqrt T$. With positive drift it grows only like $\log T$ for large $T$: the
drift eventually wins, but the longer the record, the deeper the worst drawdown in it.

### Measures built on drawdowns

- **Calmar ratio**: annualized return divided by $\mathrm{MDD}$ (usually over the last 3 years).
- **Conditional drawdown at risk (CDaR)**: the average of the worst $1-\alpha$ share of drawdowns,
  the drawdown analogue of ES (05/02).
- **Drawdown limits**: a managed account or a trader is stopped out at, say, a 10% or 20% drawdown.

## 3. Worked Examples

### Example 1: a short path by hand

Month-end fund values (CHF million): 100, 110, 99, 104.5, 88, 95, 115.

| month | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| value $W_t$ | 100 | 110 | 99 | 104.5 | 88 | 95 | 115 |
| running peak $M_t$ | 100 | 110 | 110 | 110 | 110 | 110 | 115 |
| drawdown | 0% | 0% | 10% | 5% | **20%** | 13.6% | 0% |

The maximum drawdown is 20%, from the peak of 110 in month 1 to the trough of 88 in month 4. The
fund is under water from month 2 to month 5 (four months) and sets a new peak in month 6. From the
trough, a gain of 25% (88 → 110) is needed just to get back to the peak.

### Example 2: how much gain does a loss need?

A Swiss equity index falls 45% from peak to trough (roughly the 2007–2009 fall of the SMI). The
recovery needs $0.45 / 0.55 = 82\%$. At 6.5% a year that takes $\log(1.82)/\log(1.065) \approx 9.5$ years.

### Example 3: same returns, different order

Four annual returns of +10%, +10%, −10%, −10%, in three orders:

| order | wealth path | final | MDD |
|---|---|---|---|
| +, +, −, − | 1.10, 1.21, 1.089, 0.980 | 0.980 | $1 - 0.980/1.21 =$ **19.0%** |
| +, −, +, − | 1.10, 0.99, 1.089, 0.980 | 0.980 | $1 - 0.980/1.10 =$ **10.9%** |
| −, −, +, + | 0.90, 0.81, 0.891, 0.980 | 0.980 | $1 - 0.81 =$ **19.0%** |

Same returns, same mean, same volatility, same final wealth; the maximum drawdown differs by a factor of almost two.

### Example 4: how likely is a 20% or a 50% drawdown?

Swiss equities: arithmetic mean $\mu = 6.5\%$, volatility $\sigma = 16\%$ a year, so log drift
$m = 0.065 - 0.16^2/2 = 0.0522$ and drawdown scale $\sigma^2/(2m) = 0.0256/0.1044 = 0.245$.

- Probability of **ever** falling 20% below today's level: $x = -\log 0.8 = 0.223$,
  $e^{-0.223/0.245} = e^{-0.91} \approx 40\%$.
- 50%: $x = \log 2 = 0.693$, $e^{-0.693/0.245} = e^{-2.83} \approx 6\%$.

For the 40/30/30 portfolio of module 04 ($\mu = 5.3\%$, $\sigma = 11.1\%$): $m = 0.053 - 0.0123/2 = 0.0468$, scale
$0.0123/0.0936 = 0.132$, so 20%: $e^{-1.70} \approx 18\%$, 50%: $e^{-5.27} \approx 0.5\%$.

These are probabilities measured from **one** starting point. Over a long investment life there
are many new peaks, each a fresh start, so a 20% drawdown in equities at some point is close to
certain.

## 4. Finance Application

### Where drawdowns are used

- **Fund reporting and selection**: maximum drawdown and the Calmar ratio appear on almost every
  hedge-fund and CTA factsheet. Investors screen on them.
- **Risk limits**: proprietary traders and managed accounts have drawdown stop-outs; a CTA that
  breaches a 20% drawdown often loses its mandate.
- **Pension funds and private wealth**: the question "can we live with a 30% fall?" is a
  drawdown question, and the recovery time matters as much as the depth.
- **Leverage decisions**: drawdowns grow faster than linearly with leverage (Python section),
  because the variance in the numerator of the drawdown scale $\sigma^2/(2m)$ grows like $L^2$, while the
  log drift in the denominator grows less than linearly and eventually falls.

### The usual misreadings

- "Fund A had a 12% maximum drawdown, fund B 25%, so A is less risky." If A has a 3-year
  record and B a 15-year record, the comparison is meaningless: longer records contain deeper
  drawdowns even with identical risk (Python section).
- "The maximum drawdown is the worst that can happen." It is the worst that **did** happen on one
  realized path. A different ordering of the same returns could have been much worse.
- "We are down 50%, we need 50% to get back." You need 100%.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
trading_days = 252

### Drawdown functions and Example 1

In [ ]:
def drawdown_series(wealth):
    # 1 - W_t / running peak, as decimals
    wealth = np.asarray(wealth, dtype=float)
    peak = np.maximum.accumulate(wealth)
    return 1 - wealth / peak


def max_drawdown(wealth):
    return drawdown_series(wealth).max()


def longest_underwater(wealth):
    # longest stretch of consecutive periods with drawdown > 0
    under = drawdown_series(wealth) > 1e-12
    longest = current = 0
    for u in under:
        current = current + 1 if u else 0
        longest = max(longest, current)
    return longest


values = np.array([100, 110, 99, 104.5, 88, 95, 115])
print("drawdowns:", np.round(drawdown_series(values), 3))
print(f"max drawdown: {max_drawdown(values):.1%}, longest under water: {longest_underwater(values)} months")

In [ ]:
# Example 2 and the recovery table
for d in [0.10, 0.20, 1 / 3, 0.45, 0.50, 0.75, 0.90]:
    print(f"loss {d:5.1%} -> gain needed {d / (1 - d):6.1%}")
print(f"years to recover 45% at 6.5% a year: {np.log(1 / 0.55) / np.log(1.065):.1f}")

# Example 3
for order in [[0.1, 0.1, -0.1, -0.1], [0.1, -0.1, 0.1, -0.1], [-0.1, -0.1, 0.1, 0.1]]:
    path = np.concatenate([[1.0], np.cumprod(1 + np.array(order))])
    print(order, f"final {path[-1]:.4f}, MDD {max_drawdown(path):.1%}")

### Twenty years of a balanced portfolio and of equities

Daily returns for 20 years from the multivariate Student-$t$ (4 degrees of freedom) of 05/01, with
the module-04 means and covariance. The portfolio is 40/30/30; the second series is Swiss equities
alone.

In [ ]:
def multivariate_t(n, Sigma, df, rng):
    # same generator as 05/01
    L = np.linalg.cholesky(Sigma * (df - 2) / df)
    z = rng.standard_normal((n, Sigma.shape[0])) @ L.T
    chi2 = rng.chisquare(df, size=(n, 1))
    return z / np.sqrt(chi2 / df)


assets = ["Swiss equities", "Global equities", "Swiss bonds"]
mu_annual = np.array([0.065, 0.07, 0.02])
vols = np.array([0.16, 0.17, 0.05])
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma_annual = np.diag(vols) @ corr @ np.diag(vols)
w = np.array([0.4, 0.3, 0.3])
df_t = 4

years = 20
n_days = years * trading_days
r_assets = mu_annual / trading_days + multivariate_t(n_days, Sigma_annual / trading_days, df_t, rng)
series = {"40/30/30 portfolio": r_assets @ w, "Swiss equities": r_assets[:, 0]}

summary = {}
for name, r in series.items():
    wealth = np.concatenate([[1.0], np.cumprod(1 + r)])
    dd = drawdown_series(wealth)
    trough = dd.argmax()
    peak = wealth[:trough + 1].argmax()
    summary[name] = {
        "annual return (geometric)": wealth[-1] ** (1 / years) - 1,
        "annual vol": r.std(ddof=1) * np.sqrt(trading_days),
        "max drawdown": dd.max(),
        "peak to trough (years)": (trough - peak) / trading_days,
        "longest under water (years)": longest_underwater(wealth) / trading_days,
        "Calmar (20y)": (wealth[-1] ** (1 / years) - 1) / dd.max(),
    }
pd.DataFrame(summary).round(3)

### Path dependence: shuffle the same returns

Keep the 20 years of daily equity returns, shuffle their order 2,000 times, and recompute the
maximum drawdown. Mean, volatility and final wealth are identical in every shuffle.

In [ ]:
r_eq = series["Swiss equities"]
shuffled_mdd = np.empty(2_000)
for k in range(len(shuffled_mdd)):
    path = np.cumprod(1 + rng.permutation(r_eq))
    shuffled_mdd[k] = max_drawdown(np.concatenate([[1.0], path]))

actual_mdd = summary["Swiss equities"]["max drawdown"]
print(f"actual order:  MDD {actual_mdd:.1%}")
print(f"shuffled:      median {np.median(shuffled_mdd):.1%}, 5%-95% range "
      f"{np.quantile(shuffled_mdd, 0.05):.1%} - {np.quantile(shuffled_mdd, 0.95):.1%}")

The realized maximum drawdown (38%) is one draw from a wide distribution: reordering the same
daily returns gives anything from 29% to 54% (5%–95% range). The returns themselves do not
determine it; the order does.

### Checking the Brownian-motion formulas

Simulate many 50-year paths of daily log returns (normal, so that the Brownian-motion
approximation applies) for Swiss equities and the portfolio. Compare:

1. the share of paths that ever fall 20% or 50% below the start with $e^{-2mx/\sigma^2}$;
2. the distribution of the log drawdown in the second half of each path with the exponential.

Fifty years is long enough that the probability of a first drop after year 50 is negligible for
these parameters.

In [ ]:
def simulate_log_wealth(m, sigma, years, n_paths, rng, steps_per_year=trading_days):
    dt = 1 / steps_per_year
    steps = m * dt + sigma * np.sqrt(dt) * rng.standard_normal((n_paths, years * steps_per_year))
    return np.concatenate([np.zeros((n_paths, 1)), np.cumsum(steps, axis=1)], axis=1)


sigma_p = np.sqrt(w @ Sigma_annual @ w)
mu_p = w @ mu_annual
cases = {"Swiss equities": (0.065, 0.16), "40/30/30 portfolio": (mu_p, sigma_p)}

rows = []
for name, (mu, sigma) in cases.items():
    m = mu - sigma**2 / 2
    X = simulate_log_wealth(m, sigma, 50, 4_000, rng)
    lowest = X.min(axis=1)
    for loss in [0.20, 0.50]:
        x = -np.log(1 - loss)
        rows.append({"asset": name, "drawdown": f"{loss:.0%}", "log drift m": m,
                     "scale σ²/2m": sigma**2 / (2 * m),
                     "formula": np.exp(-2 * m * x / sigma**2),
                     "simulated": np.mean(lowest <= -x)})
    if name == "Swiss equities":
        X_eq, m_eq, sigma_eq = X, m, sigma
bm_check = pd.DataFrame(rows).set_index(["asset", "drawdown"])
bm_check.round(4)

The simulated shares sit slightly **below** the formula because daily steps can jump over the
barrier between observations, where continuous Brownian motion would touch it.

### Expected maximum drawdown vs track-record length

For zero drift the theory gives $E[\mathrm{MDD}^{\log}] = 1.25\,\sigma\sqrt T$. With the equity drift the growth
slows down. Simulate 4,000 paths for each horizon from 1 to 30 years.

In [ ]:
horizons = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30])
X_long = simulate_log_wealth(m_eq, sigma_eq, horizons.max(), 4_000, rng)
X_zero = simulate_log_wealth(0.0, sigma_eq, horizons.max(), 4_000, rng)


def mdd_log(X):
    # maximum log drawdown along each row
    return (np.maximum.accumulate(X, axis=1) - X).max(axis=1)


horizon_rows = []
for T in horizons:
    n = T * trading_days + 1
    mdd_drift = mdd_log(X_long[:, :n])
    mdd_zero = mdd_log(X_zero[:, :n])
    horizon_rows.append({
        "years": T,
        "zero drift: simulated E[MDD log]": mdd_zero.mean(),
        "zero drift: 1.25 σ √T": np.sqrt(np.pi / 2) * sigma_eq * np.sqrt(T),
        "equity drift: E[MDD log]": mdd_drift.mean(),
        "equity drift: median MDD (%)": 1 - np.exp(-np.median(mdd_drift)),
        "P(MDD > 30%)": np.mean(mdd_drift > -np.log(0.7)),
    })
horizon_table = pd.DataFrame(horizon_rows).set_index("years")
horizon_table.round(3)

### Comparing funds by maximum drawdown

Two managers run **the same** strategy (Swiss equities as above). Manager A has a 3-year track
record, manager B a 15-year one. How often does A show the smaller maximum drawdown?

In [ ]:
n_A, n_B = 3 * trading_days + 1, 15 * trading_days + 1
X_A = simulate_log_wealth(m_eq, sigma_eq, 3, 4_000, rng)
X_B = simulate_log_wealth(m_eq, sigma_eq, 15, 4_000, rng)
mdd_A = 1 - np.exp(-mdd_log(X_A))
mdd_B = 1 - np.exp(-mdd_log(X_B))
calmar_A = (np.exp(X_A[:, -1] / 3) - 1) / mdd_A

print(f"median MDD, 3-year record:  {np.median(mdd_A):.1%}")
print(f"median MDD, 15-year record: {np.median(mdd_B):.1%}")
print(f"P(A's MDD < B's MDD) with identical risk: {np.mean(mdd_A < mdd_B):.0%}")
print(f"3-year Calmar ratio of the same strategy: median {np.median(calmar_A):.2f}, "
      f"5%-95% range {np.quantile(calmar_A, 0.05):.2f} to {np.quantile(calmar_A, 0.95):.2f}")

### Leverage and drawdowns

Lever the 40/30/30 portfolio by $L = 1, 1.5, 2, 3$, financing at 0.5% a year. Arithmetic mean
$L(\mu_p - r_f) + r_f$, volatility $L\sigma_p$, so the log drift is $m(L) = r_f + L(\mu_p - r_f) - L^2\sigma_p^2/2$,
a parabola in $L$.

In [ ]:
rf = 0.005
lev_rows = []
for L in [1.0, 1.5, 2.0, 3.0]:
    mu_L, sigma_L = rf + L * (mu_p - rf), L * sigma_p
    m_L = mu_L - sigma_L**2 / 2
    X_L = simulate_log_wealth(m_L, sigma_L, 20, 2_000, rng)
    mdd_L = 1 - np.exp(-mdd_log(X_L))
    lev_rows.append({"leverage": L, "annual vol": sigma_L, "log drift m": m_L,
                     "scale σ²/2m": sigma_L**2 / (2 * m_L),
                     "median 20y MDD": np.median(mdd_L),
                     "P(MDD > 50%)": np.mean(mdd_L > 0.5)})
lev_table = pd.DataFrame(lev_rows).set_index("leverage")
lev_table.round(3)

Up to 3x, leverage still **raises** the log drift (from 4.7% to 9.3% a year), so long-run wealth
grows faster. But the drawdown scale rises from 0.13 to 0.59, more than four times, and a 50% drawdown
within 20 years goes from 2% to 95% likely. Higher growth, bought with much deeper falls on the way.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
t_years = np.arange(n_days + 1) / trading_days
for name, color in [("40/30/30 portfolio", "tab:blue"), ("Swiss equities", "tab:red")]:
    wealth = np.concatenate([[1.0], np.cumprod(1 + series[name])])
    axes[0].plot(t_years, wealth, color=color, label=name)
    axes[0].plot(t_years, np.maximum.accumulate(wealth), color=color, linestyle=":", linewidth=1)
    axes[1].fill_between(t_years, -drawdown_series(wealth), color=color, alpha=0.35, label=name)
axes[0].set_yscale("log")
axes[0].set_yticks([0.75, 1, 1.5, 2, 3])
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.2f}"))
axes[0].yaxis.set_minor_formatter(plt.NullFormatter())
axes[0].set_ylabel("wealth (CHF per CHF 1, log scale)")
axes[0].set_title("Wealth and running peak (dotted), 20 simulated years")
axes[0].legend()
axes[0].grid(True, alpha=0.3, which="both")
axes[1].set_ylabel("drawdown")
axes[1].set_xlabel("years")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(shuffled_mdd, bins=60, alpha=0.8)
axes[0].axvline(actual_mdd, color="black", linestyle="--", label=f"actual order {actual_mdd:.0%}")
axes[0].set_title("MDD of the same 20 years of returns, shuffled")
axes[0].set_xlabel("maximum drawdown")
axes[0].set_ylabel("count")
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

d = np.linspace(0, 0.9, 200)
axes[1].plot(d, d / (1 - d), color="tab:red")
for loss in [0.2, 0.5, 0.75]:
    axes[1].plot(loss, loss / (1 - loss), "o", color="black")
    axes[1].annotate(f"−{loss:.0%} needs +{loss / (1 - loss):.0%}", (loss, loss / (1 - loss)),
                     textcoords="offset points", xytext=(-110, 5))
axes[1].plot(d, d, color="grey", linestyle=":", label="gain = loss")
axes[1].set_ylim(0, 5)
axes[1].set_title("Gain needed to recover a loss")
axes[1].set_xlabel("loss from peak")
axes[1].set_ylabel("gain needed")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Long-run distribution of the log drawdown, Swiss equities: second half of each 50-year path
half = X_eq.shape[1] // 2
D = (np.maximum.accumulate(X_eq, axis=1) - X_eq)[:, half::trading_days].ravel()
x_grid = np.linspace(0, 1.5, 200)
scale_eq = sigma_eq**2 / (2 * m_eq)

plt.figure(figsize=(8, 5))
D_sorted = np.sort(D)
plt.semilogy(D_sorted, 1 - np.arange(len(D_sorted)) / len(D_sorted), label="simulated")
plt.semilogy(x_grid, np.exp(-x_grid / scale_eq), "--", color="black",
             label=f"exponential, mean σ²/2m = {scale_eq:.3f}")
for loss in [0.2, 0.5]:
    plt.axvline(-np.log(1 - loss), color="grey", linestyle=":")
    plt.text(-np.log(1 - loss) + 0.01, 0.5, f"{loss:.0%} drawdown", color="grey")
plt.ylim(1e-3, 1)
plt.xlim(0, 1.5)
plt.title("Swiss equities: P(log drawdown > x) on a random day, years 25–50")
plt.xlabel("log drawdown x")
plt.ylabel("probability")
plt.grid(True, alpha=0.3, which="both")
plt.legend(loc="lower left")
plt.show()

The simulated tail lies slightly under the exponential, more so for deep drawdowns: the paths
are observed once a day, so the running peak is a little lower than the true continuous-time peak.
The same effect made the barrier probabilities above slightly smaller than the formula.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(horizons, horizon_table["zero drift: simulated E[MDD log]"], "o", color="tab:grey",
             label="zero drift, simulated")
T_grid = np.linspace(0, 30, 200)
axes[0].plot(T_grid, np.sqrt(np.pi / 2) * sigma_eq * np.sqrt(T_grid), color="tab:grey", label="1.25 σ √T")
axes[0].plot(horizons, horizon_table["equity drift: E[MDD log]"], "o-", color="tab:red",
             label=f"drift m = {m_eq:.1%}, simulated")
axes[0].set_title("Expected maximum log drawdown vs track-record length")
axes[0].set_xlabel("years")
axes[0].set_ylabel("E[max log drawdown]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(mdd_A, bins=60, range=(0, 0.8), histtype="step", linewidth=1.5, label="3-year record")
axes[1].hist(mdd_B, bins=60, range=(0, 0.8), histtype="step", linewidth=1.5, label="15-year record")
axes[1].set_title("Same strategy, different record lengths")
axes[1].set_xlabel("maximum drawdown")
axes[1].set_ylabel("count")
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Monthly values (CHF): 50, 55, 60, 48, 51, 42, 58, 63. Compute the drawdown series, the maximum
   drawdown, the trough, and the gain needed from the trough to recover.
2. A fund is down 30% from its peak. What return does it need to recover, and how many years at
   8% a year?

### Intermediate

3. Using the formula $e^{-2mx/\sigma^2}$, find the volatility at which a strategy with an arithmetic mean
   of 8% a year has a 10% probability of ever falling 30% below its starting level. (Solve
   numerically.)
4. Interpret: a factsheet shows "maximum drawdown since inception (2023): −9%, Calmar ratio 1.4"
   for a long-only equity fund. What two questions should an investor ask before comparing this
   with a fund showing "maximum drawdown since inception (2008): −48%"?

### Advanced

5. From the leverage formula $m(L) = r_f + L(\mu_p - r_f) - L^2\sigma_p^2/2$, find the leverage that maximizes
   the log drift, and the leverage at which the log drift becomes zero. What happens to the drawdown
   scale $\sigma^2/(2m)$ as $L$ approaches that point?

## 8. Solutions / Checks

In [ ]:
# 1.
v = np.array([50, 55, 60, 48, 51, 42, 58, 63])
dd = drawdown_series(v)
print("1. drawdowns:", np.round(dd, 3), f"MDD {dd.max():.1%} at month {dd.argmax()}, "
      f"gain needed {np.maximum.accumulate(v)[dd.argmax()] / v[dd.argmax()] - 1:.1%}")

# 2.
print(f"2. gain {0.3 / 0.7:.1%}, years at 8%: {np.log(1 / 0.7) / np.log(1.08):.1f}")

# 3.
from scipy import optimize
x30 = -np.log(0.7)
prob = lambda s: np.exp(-2 * (0.08 - s**2 / 2) * x30 / s**2) - 0.10
s_star = optimize.brentq(prob, 0.01, 0.39)
print(f"3. volatility with a 10% chance of ever losing 30%: {s_star:.1%}")

# 5.
L_opt = (mu_p - rf) / sigma_p**2
L_zero = max(np.roots([-sigma_p**2 / 2, mu_p - rf, rf]))
print(f"5. growth-optimal leverage {L_opt:.2f}, log drift zero at L = {L_zero:.2f}")

**1.** Peaks: 50, 55, 60, 60, 60, 60, 60, 63. The maximum drawdown is 30% (60 → 42 in month 5); a gain of
42.9% is needed from 42 back to 60. The fund recovers in month 7 (63).

**2.** $0.3/0.7 = 42.9\%$; at 8% a year, $\log(1/0.7)/\log(1.08) \approx 4.6$ years.

**3.** About 14.6% a year. Above that, a 30% fall from any given starting point becomes more likely than 10%.

**4.** (a) Track-record length: a 3-year record since 2023 contains no major equity crisis; the
2008 fund's −48% includes the financial crisis. Maximum drawdown grows with the record length even with
identical risk. (b) What market environment and what drawdown would the 2023 fund have had in 2008 or
2020 (a backtest or a comparable fund)? A Calmar ratio over a short, calm period says little.

**5.** $m'(L) = (\mu_p - r_f) - L\sigma_p^2 = 0$ gives $L^* = (\mu_p - r_f)/\sigma_p^2 \approx 3.9$ (the growth-optimal
or Kelly leverage). $m(L) = 0$ at $L \approx 7.9$. As $L$ approaches it, $m \to 0$ while $\sigma^2$ grows, so the
scale $\sigma^2/(2m) \to \infty$: deep drawdowns become near-certain. Long before that, at leverage
well below $L^*$, they are already severe: in the table above a 50% drawdown within 20 years is more
likely than not at 2x (57%) and nearly certain at 3x (95%), although both still raise the log drift.

## 9. Conclusion

- Drawdown is the loss from the running peak; maximum drawdown is the worst peak-to-trough loss on a path.
- A loss of $d$ needs a gain of $d/(1-d)$ to recover: 50% needs 100%.
- Drawdowns depend on the order of returns; volatility, VaR and ES do not.
- For log wealth with drift $m$ and volatility $\sigma$, the drawdown scale is $\sigma^2/(2m)$; the probability of ever falling $x$ in log terms is $e^{-2mx/\sigma^2}$.
- Expected maximum drawdown grows with the record length ($\sqrt T$ with zero drift, $\log T$ with positive drift), so records of different lengths are not comparable.
- Leverage multiplies the variance by $L^2$ but the log drift by much less, so drawdowns grow much faster than leverage, even where leverage still raises long-run growth.

Next: [04 Risk contributions and Euler allocation](./04_risk_contributions_and_euler_allocation.ipynb).